In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/activity_labels.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/README.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/features_info.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/features.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/subject_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/X_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_acc_y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/total_acc_y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/total_acc_z_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_acc_z_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_gyr

# Loading and Normalizing the dataset

In [2]:
import numpy as np

def load_raw_signals(folder_path, filenames):
    signals = []
    for name in filenames:
        file_path = f"{folder_path}/{name}.txt"
        data = np.loadtxt(file_path)
        signals.append(data)
    
    return np.transpose(np.array(signals), (1, 2, 0))
filenames = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z'
]
filenames_train = [f + '_train' for f in filenames]
filenames_test = [f + '_test' for f in filenames]
raw_data_path_train = '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/Inertial Signals'
raw_data_path_test =  '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals'

x_train_raw = load_raw_signals(raw_data_path_train, filenames_train)
X_test_raw = load_raw_signals(raw_data_path_test, filenames_test)


mean = np.mean(x_train_raw, axis=(0, 1))
std = np.std(x_train_raw, axis=(0, 1))

#normalizing 
x_train = (x_train_raw - mean) / (std + 1e-8)
X_test = (X_test_raw - mean) / (std + 1e-8) 

# loading labels
y_train = np.loadtxt('/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/y_train.txt', dtype=int) - 1
y_test = np.loadtxt('/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/y_test.txt', dtype=int) - 1

print(f"Normalized x_train shape: {x_train.shape}")
print(f"Normalized X_test shape: {X_test.shape}")

Normalized x_train shape: (7352, 128, 6)
Normalized X_test shape: (2947, 128, 6)


# Coding the 1D-CNN Architecture

In [3]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers, models
model = keras.Sequential(name="HAR_1DCNN") 
model.add(layers.Input(shape=(128, 6))) # loading an empty stack to which we'll add layers
model.add(layers.Conv1D(   #Now we're adding layers to this 
    filters=64,    #this arg defines how many local features we'll look for
    kernel_size=5, #this is essentially our window size which we slide over our 128 timesteps to extract features
    activation='relu', #rectified linear unit introduces non-linearity basically acting as a gate to decide which information is important enough to be passed to the next layer
    padding= 'same',   #This makes sure that our output is the same length as our input adds zeros so the kernel can center itself
    #input_shape=(128,6) #This defines the size of our input passed 
))
#Stacking another layer cause we want a deeper model 
model.add(layers.MaxPooling1D(pool_size=2)) 
model.add(layers.Dropout(rate=0.5)) 

model.add(layers.Conv1D(   
    filters=128,   
    kernel_size=3,
    activation='relu', 
    padding= 'same',  
))
model.add(layers.MaxPooling1D(pool_size=2)) # makes model invincible to temporal shifts (like a pattern that occurs at timestep 11 and 12 are for the same motion provided the pattern is similar)
model.add(layers.Dropout(rate=0.5)) # turns off 50% of neurons mid training to make sure the model doesnt memorize 
model.add(layers.GlobalAveragePooling1D()) # converts our 2D matrix (timesteps x features) into one long 1D vector


model.add(layers.Dense(32, activation='relu')) #input neurons


model.add(layers.Dropout(0.5)) #turns off 50% of neurons mid training to prevent memorization


model.add(layers.Dense(6, activation='softmax')) #softmax is a function that calculates probabilities of each of 6 outcomes and chooses the highest probability option
model.compile(
    optimizer='adam',  # updates weights and bias
    loss='sparse_categorical_crossentropy', #our loss function
    metrics=['accuracy'] # what metrics we'll be seeing
)

model.summary() # summary

history = model.fit(    #actual training logic epochs means training cycles btw
    x_train, y_train,
    epochs=30, # this just means the model will run 30 epochs
    batch_size=16, # the model will look at 16 samples at ones
    validation_data=(X_test, y_test), #validates accuracy against test data
    shuffle=True,   # shuffling the data
)

model.save('my_model.h5')

2026-05-16 17:26:06.639525: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778952367.000209      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778952367.098580      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778952368.025629      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778952368.025677      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778952368.025680      23 computation_placer.cc:177] computation placer alr

Model: "HAR_1DCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 128, 64)        │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 64, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,014 (121.15 KB)

 Trainable params: 31,014 (121.15 KB)

...

Epoch 1/30


I0000 00:00:1778952408.137596      71 service.cc:152] XLA service 0x7933a800a510 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778952408.137627      71 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778952408.137631      71 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778952408.586533      71 cuda_dnn.cc:529] Loaded cuDNN version 91002


 63/460 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2107 - loss: 1.7183

I0000 00:00:1778952412.217939      71 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


460/460 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.3546 - loss: 1.4019 - val_accuracy: 0.6274 - val_loss: 0.6933
Epoch 2/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5916 - loss: 0.7618 - val_accuracy: 0.6610 - val_loss: 0.6437
Epoch 3/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6276 - loss: 0.6870 - val_accuracy: 0.6461 - val_loss: 0.6581
Epoch 4/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6395 - loss: 0.6471 - val_accuracy: 0.6685 - val_loss: 0.6256
Epoch 5/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6396 - loss: 0.6446 - val_accuracy: 0.6966 - val_loss: 0.6210
Epoch 6/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6655 - loss: 0.6233 - val_accuracy: 0.7072 - val_loss: 0.6270
Epoch 7/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6904 - loss: 0.6098 - val_accuracy: 0.7611 - val_loss: 0.5679
Epoch 8/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7178 - loss: 0.5649 - val_accuracy: 0.7967 - va

# Converting the .h5 model to .tflite model

In [4]:
import tensorflow as tf
model=tf.keras.models.load_model('my_model.h5') # loads the saved model

converter=tf.lite.TFLiteConverter.from_keras_model(model) # initializes the converter
tflite_fp32model=converter.convert() # does the converting here 
with open('model_fp32.tflite','wb') as f: # writing the new tflite model to a file named 'model_fp32.tflite' btw wb means write binary
    f.write(tflite_fp32model)

INFO:tensorflow:Assets written to: /tmp/tmpds1gq8et/assets


INFO:tensorflow:Assets written to: /tmp/tmpds1gq8et/assets


Saved artifact at '/tmp/tmpds1gq8et'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 6), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  133266383436880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383436496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383433616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383436112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133269026762256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383434768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383427856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383431696: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1778952464.666183      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778952464.666229      23 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1778952464.672562      23 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


# Quantizing the FP32 model to INT8 for edge deployment 

In [5]:
import numpy as np
import tensorflow as tf
model=tf.keras.models.load_model('my_model.h5') #loads the original model

def representative_data_gen():  #this function creates a representive dataset for the int8 model
    
    for input_value in x_train[:500]: #loops through 100 samples of the actual dataset
        yield [np.array(input_value, dtype=np.float32)[np.newaxis, ...]] # Ensure the sample has the correct batch (new.axis) dimension [1, ...] and type (dtype)
#the next two lines configures the converter for full int8 quantization
converter.optimizations=[tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops= [tf.lite.OpsSet.TFLITE_BUILTINS_INT8] #This line tells that all the operations must be quantized to int8
#the next two lines set the input and output to int8
converter.inference_input_type=tf.int8
converter.inference_output_type=tf.int8
tflite_int8model= converter.convert() #actual conversion happen here
with open('model_int8.tflite', 'wb') as f: # writes the new model to a file named 'model_int8.tflite'
    f.write(tflite_int8model)

INFO:tensorflow:Assets written to: /tmp/tmp5c9f6n2i/assets


INFO:tensorflow:Assets written to: /tmp/tmp5c9f6n2i/assets


Saved artifact at '/tmp/tmp5c9f6n2i'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 6), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  133266383436880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383436496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383433616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383436112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133269026762256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383434768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383427856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133266383431696: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1778952465.409496      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778952465.409520      23 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


# INT8 evaluation script I generated using AI 

In [6]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# ==========================================
# 1. LOAD THE INT8 MODEL & ALLOCATE TENSORS
# ==========================================
model_path = "model_int8.tflite"
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

# Get structural details for inputs and outputs
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# ==========================================
# 2. EXTRACT QUANTIZATION PARAMETERS (S & Z)
# ==========================================
input_scale, input_zero_point = input_details[0]['quantization']
output_scale, output_zero_point = output_details[0]['quantization']

print("====== MODEL QUANTIZATION INFO ======")
print(f"Input Type:   {input_details[0]['dtype']}")
print(f"Output Type:  {output_details[0]['dtype']}")
print(f"Input Scale:  {input_scale} | Zero-Point: {input_zero_point}")
print(f"Output Scale: {output_scale} | Zero-Point: {output_zero_point}")
print("=====================================\n")

# ==========================================
# 3. RUN INFERENCE LOOP OVER THE TEST SET
# ==========================================
quant_predictions = []

print("Running inference on test dataset...")
# Loop through every sequential sample in your 1D-CNN test array
for sample in X_test:
    
    # A. Scale raw floating-point input to fixed-point INT8
    # Formula: q = round(r / S) + Z
    q_input = np.round(sample / input_scale) + input_zero_point
    
    # B. Clamp values to ensure they stay within valid 8-bit bounds [-128, 127]
    q_input = np.clip(q_input, -128, 127).astype(np.int8)
    
    # C. Add the explicit batch dimension required by TFLite [1, sequence_length, features]
    q_input = np.expand_dims(q_input, axis=0)
    
    # D. Pass the processed INT8 array into the interpreter tensor slot
    interpreter.set_tensor(input_details[0]['index'], q_input)
    
    # E. Run the network math natively in INT8
    interpreter.invoke()
    
    # F. Extract the resulting raw integer output tensor
    q_output = interpreter.get_tensor(output_details[0]['index'])
    
    # G. De-quantize the INT8 output back to Float space for evaluation
    # Formula: r = S * (q - Z)
    f_output = (q_output.astype(np.float32) - output_zero_point) * output_scale
    
    # Append prediction (removing the temporary batch dimension)
    quant_predictions.append(f_output[0])

# Convert list of evaluations into a clean NumPy array
quant_predictions = np.array(quant_predictions)

# ==========================================
# 4. POST-PROCESS & CALCULATE ACCURACY
# ==========================================

# Parse predicted probabilities into discrete class indices
if len(quant_predictions.shape) > 1 and quant_predictions.shape[1] > 1:
    # Multi-class classification (Categorical Cross-Entropy)
    final_pred_classes = np.argmax(quant_predictions, axis=1)
    true_classes = np.argmax(y_test, axis=1) if len(y_test.shape) > 1 else y_test


# ==========================================
# 5. PRINT THE FINAL METRICS DISPLAY
# ==========================================
print("\n" + "="*45)
print("   INT8 1D-CNN TFLITE EVALUATION METRICS   ")
print("="*45)

accuracy = accuracy_score(true_classes, final_pred_classes) * 100
print(f"Overall Accuracy on Test Set: {accuracy:.2f}%\n")

print("Detailed Performance Across Classes:")
print(classification_report(true_classes, final_pred_classes))

print("Confusion Matrix:")
print(confusion_matrix(true_classes, final_pred_classes))
print("="*45)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


====== MODEL QUANTIZATION INFO ======
Input Type:   <class 'numpy.int8'>
Output Type:  <class 'numpy.int8'>
Input Scale:  0.06161845102906227 | Zero-Point: 13
Output Scale: 0.00390625 | Zero-Point: -128

Running inference on test dataset...

   INT8 1D-CNN TFLITE EVALUATION METRICS   
Overall Accuracy on Test Set: 88.16%

Detailed Performance Across Classes:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       496
           1       0.99      0.94      0.96       471
           2       0.95      0.99      0.97       420
           3       0.66      0.90      0.76       491
           4       0.86      0.88      0.87       532
           5       0.97      0.63      0.76       537

    accuracy                           0.88      2947
   macro avg       0.90      0.89      0.89      2947
weighted avg       0.90      0.88      0.88      2947

Confusion Matrix:
[[496   0   0   0   0   0]
 [  3 444  24   0   0   0]
 [  0   6 414   0   0   